# 2. Calidad y preparación de datos
**RetailNova — Caso 1 (SCY1102 Gestión de Proyectos de Datos, Sección 001D)** · Hurtubia · Pacheco · Curi · Gómez

Diagnóstico de nulos e inconsistencias, corrección de fechas, imputación justificada, validación formal y feature engineering RFM. Código en `src/pipeline_datos.py`. No se modifica `data/raw/`; el resultado se escribe en `data/processed/`.

## 2.1 Configuración

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # permite importar src/ desde notebooks/

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config as cfg

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.3f}".format

In [2]:
from src.pipeline_datos import (diagnostico_nulos, corregir_coherencia_temporal, imputar_nulos,
                                validar_dataset, agregar_rfm)
from src.generador_dataset import cargar_estado_rng

df = pd.read_csv(cfg.CSV_12M)
rng = cargar_estado_rng(cfg.ESTADO_RNG)   # continúa la secuencia aleatoria del cuaderno 01
print(df.shape)

(51346, 17)


## 2.2 Diagnóstico inicial

In [3]:
nulos_iniciales = diagnostico_nulos(df)
nulos_iniciales

,Nulos,Porcentaje (%)
id_cliente,0,0.000
edad,2039,3.970
comuna,2080,4.050
region,0,0.000
fecha_registro,0,0.000
canal_compra,525,1.020
cantidad_compras,0,0.000
frecuencia_compra,0,0.000
monto_promedio_compra,1070,2.080
monto_total_acumulado,0,0.000


## 2.3 Coherencia temporal
Un cliente no puede comprar antes de registrarse. Se corrige moviendo `fecha_registro` hacia atrás (0 a 59 días antes de la última compra) **sin alterar** la definición de abandono (180 días de inactividad).

In [4]:
corregidos = corregir_coherencia_temporal(df, rng)
print(f"Registros corregidos: {corregidos}")

Registros corregidos: 11033


## 2.4 Imputación justificada
- `edad`: mediana (robusta a atípicos).
- `canal_compra`: moda (categoría más frecuente).
- `monto_promedio_compra`: mediana **por canal** (el monto depende del canal).
- `comuna`: aleatoria válida dentro de la misma región (se evita asignar comunas de otra región).

In [5]:
imputar_nulos(df, rng)
comparativa = pd.DataFrame({"Nulos iniciales": nulos_iniciales["Nulos"], "Nulos finales": df.isnull().sum()})
comparativa[(comparativa > 0).any(axis=1)]

,Nulos iniciales,Nulos finales
edad,2039,0
comuna,2080,0
canal_compra,525,0
monto_promedio_compra,1070,0


## 2.5 Validaciones formales
`validar_dataset` lanza un error con el motivo si algo falla: nulos, fechas, comunas fuera de región, edades fuera de 18–80, tasa de abandono fuera de 20–28 %, IDs duplicados.

In [6]:
validar_dataset(df)
print("Todas las validaciones de calidad pasaron.")

Todas las validaciones de calidad pasaron.


## 2.6 Feature engineering (RFM)
Recencia, antigüedad y tasas por compra (plan EP3, actividad 1.4.3).

**Atención con la fuga de datos:** `recencia_dias` es exactamente la regla que define el abandono (> 180 días). Si entra al modelo, este "memoriza" la etiqueta. Se calcula para análisis, pero **no** se usa como predictora.

In [7]:
df = agregar_rfm(df)
df[["recencia_dias", "antiguedad_dias", "tasa_reclamos", "tasa_devoluciones"]].describe().T

,count,mean,std,min,25%,50%,75%,max
recencia_dias,"51,346.000",131.495,90.719,0.000,59.000,118.000,177.000,349.000
antiguedad_dias,"51,346.000",245.727,78.225,30.000,183.000,247.000,315.000,407.000
tasa_reclamos,"51,346.000",0.168,0.374,0.000,0.000,0.000,0.143,4.000
tasa_devoluciones,"51,346.000",0.029,0.118,0.000,0.000,0.000,0.000,2.000


In [8]:
fuga = df.groupby("abandono")["recencia_dias"].agg(["min", "max"])
fuga.index = ["Activo", "Abandona"]
fuga  # los rangos no se solapan: recencia separa perfectamente la etiqueta

,min,max
Activo,0,179
Abandona,181,349


## 2.7 Exportación

In [9]:
for c in ("fecha_registro", "fecha_ultima_compra"):
    df[c] = pd.to_datetime(df[c]).dt.strftime("%Y-%m-%d")
cfg.CSV_PREPARADO.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(cfg.CSV_PREPARADO, index=False)
print(f"{len(df):,} filas · 0 nulos · {cfg.CSV_PREPARADO.name}")

51,346 filas · 0 nulos · retailnova_clientes_12m_completo.csv


**Siguiente:** `03_modelado.ipynb`.